cleaning & feature engineering 

In [31]:
import pandas as pd
import numpy as np
import ast
import re
import os

In [2]:
#loading the data
df = pd.read_csv('../Data/data.csv')

In [3]:
df_clean = df.copy()

In [4]:
# Drop dead columns
# We use errors='ignore' just in case you run the cell twice
df_clean = df_clean.drop(columns=['delivery_details', 'delivery_fee'], errors='ignore')

In [5]:
print(f"Original columns: {len(df.columns)}")
print(f"Cleaned columns: {len(df_clean.columns)}")
print("Dead columns dropped. Ready for Task 2 (Brand Recovery).")

Original columns: 22
Cleaned columns: 20
Dead columns dropped. Ready for Task 2 (Brand Recovery).


brand recovery

In [7]:
# Create a mask to see rows that originally had no brand 
# (Assuming 'brand' in raw data was NaN for these)
originally_missing = df['brand'].isna()

print(f"Total brands recovered: {originally_missing.sum()}")
print("-" * 50)

# Spot-check 20 random samples from the recovered group
check_df = df_clean[originally_missing][['title', 'brand']].sample(20, random_state=42)

# Styling the output for easier manual reading
display(check_df)

# Validation logic
remaining_nulls = df_clean['brand'].isnull().sum()
null_percentage = (remaining_nulls / len(df_clean)) * 100

print("-" * 50)
print(f"Current Brand Null Count: {remaining_nulls} ({null_percentage:.2f}%)")
if null_percentage < 5:
    print(" SUCCESS: Brand nulls are below the 5% target!")
else:
    print(" WARNING: Still above 5% nulls. Add more brands to KNOWN_BRANDS.")

Total brands recovered: 893
--------------------------------------------------


,title,brand
1483,XMobile Dabang Plus,NaN
1213,WS27 Bluetooth Calling Watch,NaN
1298,TG-38 Ultra Smartwatch,NaN
1494,Dcode Cygnal 2 Lite,NaN
812,Samsung Galaxy Buds Pro,NaN
1063,QCY T1C TWS Bluetooth Earphones,NaN
1073,Audionic Bluetooth Neckband (B730),NaN
1106,1More Omthing PistonBuds TWS Bluetooth Earbuds,NaN
981,Audionic Signature Premium Neckband (N220),NaN
909,HOTTU TWS Earpods (P73 Max),NaN


--------------------------------------------------
Current Brand Null Count: 893 (53.60%)


In [8]:
KNOWN_BRANDS = [
    # Multi-word brands first
    'Red Magic', 'VGO TEL', 'Dcode Cygnal', 'Club Mobile', 'Mobile Dabang', 
    '1More Omthing', '1More',
    # Single word brands
    'Samsung', 'Apple', 'Xiaomi', 'Oppo', 'Vivo', 'Realme', 'OnePlus', 
    'Nokia', 'Google', 'Infinix', 'Tecno', 'Huawei', 'Motorola', 'Lenovo', 
    'HP', 'Dell', 'Asus', 'Acer', 'MSI', 'Razer', 'Sony', 'JBL', 'Anker', 
    'QCY', 'Sennheiser', 'Bose', 'Nothing', 'Sparx', 'Hisense', 'Audionic', 
    'Ronin', 'Joyroom', 'Baseus', 'HOTTU', 'Calme', 'Ansty', 'WS27', 'Dcode'
]

In [9]:
def recover_brand(row):
    # Check if existing brand is valid
    existing = str(row['brand']).strip()
    if pd.notnull(row['brand']) and existing.lower() != 'nan' and existing != '':
        return row['brand']
    
    title = str(row['title'])
    
    # Logic A: Priority Scan for Known Brands
    for brand in KNOWN_BRANDS:
        if brand.lower() in title.lower():
            return brand
            
    # Logic B: Fallback - First word if it starts with a Capital
    match = re.search(r'\b[A-Z][a-zA-Z0-9]*\b', title)
    if match:
        return match.group(0)
    
    return "Unknown" # Never return actual NaN/Null here

# APPLY AND CHECK
df_clean['brand'] = df_clean.apply(recover_brand, axis=1)

# Verification
new_null_count = df_clean['brand'].isna().sum()
unknown_count = (df_clean['brand'] == 'Unknown').sum()
print(f"New Null Count: {new_null_count}")
print(f"Unknowns (Fallback failed): {unknown_count}")

New Null Count: 0
Unknowns (Fallback failed): 0


specification JSON parsing

In [10]:
# 1. Standardize the Extraction Logic
def extract_gb_value(value):
    if not value or pd.isna(value): return None
    # This regex finds numbers and the unit (GB or TB)
    # It handles '8GB', '8 GB', '8gb', '1TB' etc.
    match = re.search(r'(\d+)\s*(GB|TB|gb|tb|Gb|Tb)', str(value))
    if match:
        num = int(match.group(1))
        unit = match.group(2).upper()
        if unit == 'TB':
            num *= 1024  # Standardize TB to GB
        return num
    return None

In [12]:
# 1. Configuration: Expanded Aliases to catch more data
KEY_ALIASES = {
    # RAM variants
    'Memory': 'RAM', 'Installed RAM': 'RAM', 'System Memory': 'RAM', 
    'Ram': 'RAM', 'RAM Type': 'RAM', 'Standard Memory': 'RAM',
    # Storage variants
    'Hard Drive Capacity': 'Storage', 'Storage Capacity': 'Storage', 
    'SSD': 'Storage', 'HDD': 'Storage', 'Internal Storage': 'Storage', 
    'Capacity': 'Storage', 'Rom': 'Storage', 'Internal Memory': 'Storage'
}

In [13]:
# 2. Helper: Convert JSON string to clean Dictionary
def parse_specs_to_dict(spec_str):
    try:
        if pd.isna(spec_str) or str(spec_str).strip() == "" or spec_str == "{}":
            return {}
        # Convert string representation of dict to actual Python dict
        raw_dict = ast.literal_eval(spec_str)
        if not isinstance(raw_dict, dict): return {}
        
        # Normalize keys based on aliases
        normalized_dict = {}
        for k, v in raw_dict.items():
            std_key = KEY_ALIASES.get(k, k)
            normalized_dict[std_key] = v
        return normalized_dict
    except:
        return {}

In [14]:
# 3. Helper: Extract Number from strings like "16GB" or "1 TB"
def extract_gb_value(value):
    if not value or pd.isna(value): return None
    # Regex to find numbers and units (GB/TB)
    match = re.search(r'(\d+)\s*(GB|TB|gb|tb|Gb|Tb)', str(value))
    if match:
        num = int(match.group(1))
        unit = match.group(2).upper()
        if unit == 'TB':
            num *= 1024  # Standardize TB to GB
        return num
    return None

In [16]:
# --- EXECUTION ---

# Create the spec_dict column first (Fixes your KeyError)
df_clean['spec_dict'] = df_clean['specifications'].apply(parse_specs_to_dict)

# Now extract numeric values from that dict
df_clean['ram_gb'] = df_clean['spec_dict'].apply(lambda d: extract_gb_value(d.get('RAM')))
df_clean['storage_gb'] = df_clean['spec_dict'].apply(lambda d: extract_gb_value(d.get('Storage')))

# --- SUCCESS METRIC CHECK ---

relevant_cats = ['Laptop', 'Mobile']
subset = df_clean[df_clean['category'].isin(relevant_cats)]

ram_fill = (subset['ram_gb'].notnull().sum() / len(subset)) * 100
storage_fill = (subset['storage_gb'].notnull().sum() / len(subset)) * 100

print(f"--- Fill Rate Report ({len(subset)} Products) ---")
print(f"RAM Fill Rate: {ram_fill:.2f}%")
print(f"Storage Fill Rate: {storage_fill:.2f}%")

if ram_fill > 80:
    print(" SUCCESS: Target met! Your RAM extraction is high quality.")
else:
    print(" WARNING: Target missed. You might need more aliases for different vendors.")

# Spot check 10 rows
display(subset[['title', 'ram_gb', 'storage_gb']].sample(min(10, len(subset))))

--- Fill Rate Report (1097 Products) ---
RAM Fill Rate: 81.22%
Storage Fill Rate: 43.48%
 SUCCESS: Target met! Your RAM extraction is high quality.


,title,ram_gb,storage_gb
1507,me Mobile Magic Sound,NaN,NaN
29,OnePlus Ace 2 12GB Ram 256GB Storage Non PTA 5G,12.0,NaN
397,HP Victus Gaming Laptop 15-FA0031DX,8.0,NaN
447,Acer Predator Helios Neo 16 PHN16-71-75FC Gami...,16.0,NaN
41,ONEPLUS 11 16GB RAM 512GB Storage Non PTA,16.0,NaN
1515,GRESSO Turbo 2,NaN,NaN
2,Tecno Spark 10,4.0,NaN
1555,Tecno Camon 19 Pro,8.0,128.0
482,HP EliteBook 840 G5 Business Laptop - Intel Co...,16.0,NaN
629,Microsoft Surface Pro 9 12th Gen Core i7 16GB ...,16.0,NaN


In [17]:
def rescue_storage_from_title(row):
    # If we already have storage from specs, keep it
    if pd.notnull(row['storage_gb']):
        return row['storage_gb']
    
    title = str(row['title'])
    
    # Regex to find storage (e.g., 128GB, 256 GB, 1TB) 
    # Specifically looking for patterns after "Ram" or standalone
    # We exclude the RAM figure by looking for common storage sizes
    matches = re.findall(r'(\d+)\s*(GB|TB|gb|tb)', title)
    
    if matches:
        # We take the largest number found in the title for storage 
        # (Since RAM is usually smaller than Storage)
        values = []
        for num, unit in matches:
            val = int(num)
            if unit.upper() == 'TB': val *= 1024
            values.append(val)
        
        # Heuristic: Storage is almost always > 16GB for modern devices 
        # while RAM is often <= 16GB.
        storage_options = [v for v in values if v > 16]
        return max(storage_options) if storage_options else max(values)
        
    return None

In [18]:
# Apply the rescue
df_clean['storage_gb'] = df_clean.apply(rescue_storage_from_title, axis=1)

# Re-check Fill Rate
storage_fill = (df_clean[df_clean['category'].isin(['Laptop', 'Mobile'])]['storage_gb'].notnull().sum() / len(subset)) * 100
print(f"New Storage Fill Rate: {storage_fill:.2f}%")

New Storage Fill Rate: 67.91%


In [20]:
def final_storage_rescue(row):
    if pd.notnull(row['storage_gb']): return row['storage_gb']
    
    title = str(row['title']).upper()
    # Look for patterns like 128GB, 256G, 1TB, 512 GB
    match = re.search(r'(\d+)\s*(GB|TB|G|T)', title)
    
    if match:
        val = int(match.group(1))
        unit = match.group(2)
        if 'T' in unit: val *= 1024
        # Filter out obvious RAM numbers (like 4, 8, 16) if they are small
        # In phones/laptops, storage is almost always 32, 64, 128, 256, 512, 1024
        if val >= 32: return float(val)
        
    return None

df_clean['storage_gb'] = df_clean.apply(final_storage_rescue, axis=1)
print(f"Final Storage Fill Rate: {(df_clean[df_clean['category'].isin(['Laptop', 'Mobile'])]['storage_gb'].notnull().sum() / 1097 * 100):.2f}%")

Final Storage Fill Rate: 74.29%


Price Normalisation

In [21]:
# 1. Fill missing original_price with Category Median
# transform() ensures we don't accidentally use a Laptop price to fill an Earbud null
df_clean['original_price'] = df_clean.groupby('category')['original_price'].transform(lambda x: x.fillna(x.median()))

# 2. Fill missing discounted_price with original_price (no sale = price stays same)
df_clean['discounted_price'] = df_clean['discounted_price'].fillna(df_clean['original_price'])

# 3. Create discount_pct and clip at 0 (Prevents errors where original < discounted)
df_clean['discount_pct'] = ((df_clean['original_price'] - df_clean['discounted_price']) / df_clean['original_price'] * 100)
df_clean['discount_pct'] = df_clean['discount_pct'].clip(lower=0).round(1)

# 4. Add 'is_coming_soon' flag
# This helps the UI (Streamlit) later to hide 'Buy Now' buttons for these items
df_clean['is_coming_soon'] = df_clean['availability'].str.contains('Coming Soon', case=False, na=False)

# Optional: For 'Coming Soon' items, ensure price logic doesn't mess up search
# We keep the median price for searching, but the flag tells the user it's not out yet.

print(f"Price cleanup complete. Missing original_prices: {df_clean['original_price'].isnull().sum()}")
print(f"Average discount found: {df_clean['discount_pct'].mean():.2f}%")
display(df_clean[['title', 'original_price', 'discounted_price', 'discount_pct', 'is_coming_soon']].sample(5))

Price cleanup complete. Missing original_prices: 0
Average discount found: 7.19%


,title,original_price,discounted_price,discount_pct,is_coming_soon
46,Samsung Galaxy S23 Ultra 12GB RAM 512GB Storag...,339999.0,339999.0,0.0,False
1530,GFive G550 Power,3599.0,3599.0,0.0,False
176,Lenovo V15 G3 Core i3 12th Generation 4GB RAM ...,109999.0,109999.0,0.0,False
292,Acer Aspire A515 57G 73HX Core i7 12th Generat...,202999.0,202999.0,0.0,False
306,HP 15S FQ5098TU Core i5 12th Generation 8GB RA...,178999.0,178999.0,0.0,False


Text Preparation for Search & Image URL Extraction 

In [ ]:
# 1. Create 'first_img' (Task 6)
def get_first_img(img_str):
    try:
        img_list = ast.literal_eval(img_str)
        return img_list[0] if (isinstance(img_list, list) and len(img_list) > 0) else "placeholder.jpg"
    except:
        return "placeholder.jpg"

df_clean['first_img'] = df_clean['imgs'].apply(get_first_img)

# 2. Create 'clean_content' (Task 5) - This fixes your KeyError!
def build_search_content(row):
    # Combine ingredients for the "Search Soup"
    ram = f"{row['ram_gb']}gb" if pd.notnull(row['ram_gb']) else ""
    storage = f"{row['storage_gb']}gb" if pd.notnull(row['storage_gb']) else ""
    
    content = f"{row['brand']} {row['category']} {row['title']} {ram} {storage}"
    content = content.lower()
    
    # Scrub the noise
    noise = ['pta approved', 'non pta', 'price in pakistan', 'official warranty', 'non-pta']
    for phrase in noise:
        content = content.replace(phrase, '')
        
    return " ".join(content.split())

df_clean['clean_content'] = df_clean.apply(build_search_content, axis=1)

# 3. Standardize Availability (Task 7)
df_clean['availability'] = df_clean['availability'].replace({'Avaiable': 'Available'}).fillna('Unknown')

print(" Columns created: 'first_img', 'clean_content', 'availability'")

✅ Columns created: 'first_img', 'clean_content', 'availability'


In [24]:
def validation_search(query, df):
    query = query.lower()
    # Now this column exists!
    results = df[df['clean_content'].str.contains(query)]
    
    print(f"Results for: '{query}'")
    print(f"Total found: {len(results)}")
    print(f"Vendors detected: {results['vendor'].unique()}")
    print("-" * 50)
    
    return results[['vendor', 'brand', 'title', 'ram_gb']].head(10)

# Run the test
validation_search("samsung 8gb", df_clean)

Results for: 'samsung 8gb'
Total found: 0
Vendors detected: []
--------------------------------------------------


,vendor,brand,title,ram_gb


In [25]:
# Diagnostic: See what the computer actually sees
print("First 5 rows of clean_content:")
print(df_clean['clean_content'].head().values)

print("\nChecking for 'Samsung' in the brand column:")
print(df_clean[df_clean['brand'].str.contains('Samsung', case=False, na=False)]['brand'].count())

print("\nChecking for 8.0 in ram_gb:")
print(df_clean[df_clean['ram_gb'] == 8.0]['ram_gb'].count())

First 5 rows of clean_content:
['nothing mobile nothing phone 1 8gb ram 256gb storage 5g black 8.0gb 256.0gb'
 'oppo mobile oppo f21 pro 8gb ram 128gb storage 5g 8.0gb 128.0gb'
 'tecno mobile tecno spark 10 4.0gb' 'vivo mobile vivo v27 5g 8.0gb'
 'apple mobile apple iphone 15 pro max 8.0gb']

Checking for 'Samsung' in the brand column:
77

Checking for 8.0 in ram_gb:
374


In [27]:
def nuclear_clean(row):
    # 1. Start with the basics
    text = f"{row['brand']} {row['category']} {row['title']}"
    
    # 2. Add RAM/Storage but strip the decimals immediately
    if pd.notnull(row['ram_gb']):
        text += f" {int(row['ram_gb'])}gb"
    if pd.notnull(row['storage_gb']):
        text += f" {int(row['storage_gb'])}gb"
    
    # 3. Lowercase and remove punctuation (including dots!)
    text = text.lower()
    text = text.replace('.', '') # This turns 8.0gb into 80gb? No, let's be safer.
    
    # 4. Better: replace '.0' specifically
    text = text.replace('.0', '')
    
    # 5. Remove marketing noise
    noise = ['pta approved', 'non pta', 'warranty', 'price in pakistan']
    for n in noise:
        text = text.replace(n, '')
        
    return " ".join(text.split())

df_clean['clean_content'] = df_clean.apply(nuclear_clean, axis=1)

# FINAL VERIFICATION - Let's look at Samsung specifically
samsung_check = df_clean[df_clean['brand'].str.contains('Samsung', case=False, na=False)]
print(f"Total Samsung rows: {len(samsung_check)}")
print("Sample content for a Samsung row:")
if len(samsung_check) > 0:
    print(samsung_check['clean_content'].iloc[0])

# Try the search one more time
final_test = df_clean[df_clean['clean_content'].str.contains("samsung")]
print(f"\nSearch for 'samsung' found: {len(final_test)}")

final_test_8gb = df_clean[df_clean['clean_content'].str.contains("samsung") & df_clean['clean_content'].str.contains("8gb")]
print(f"Search for 'samsung' AND '8gb' found: {len(final_test_8gb)}")

Total Samsung rows: 77
Sample content for a Samsung row:
samsung mobile samsung galaxy s23 ultra 12gb ram 512gb storage 12gb 512gb

Search for 'samsung' found: 77
Search for 'samsung' AND '8gb' found: 33


In [28]:
# Task 7 Logic Snippet
status_map = {
    'Avaiable': 'Available', 
    'Available': 'Available',
    'Out of stock': 'Out of Stock',
    'Coming Soon': 'Coming Soon'
}
df_clean['availability'] = df_clean['availability'].map(status_map).fillna('Unknown')

Validate & Export 

In [29]:
def check_kpis(df):
    print("---  FINAL DATA QUALITY AUDIT 🏁 ---")
    
    # KPI 1: Brand Nulls
    brand_fail = df['brand'].isna().sum()
    kpi1 = "PASS" if brand_fail == 0 else f"FAIL ({brand_fail} nulls)"
    
    # KPI 2: Price Nulls
    price_fail = df['original_price'].isna().sum()
    kpi2 = "PASS" if price_fail == 0 else f"FAIL ({price_fail} nulls)"
    
    # KPI 3: Mobile/Laptop Fill Rate (>80%)
    relevant = df[df['category'].isin(['Mobile', 'Laptop'])]
    ram_rate = (relevant['ram_gb'].notnull().sum() / len(relevant)) * 100
    kpi3 = "PASS" if ram_rate >= 80 else f"FAIL ({ram_rate:.2f}%)"
    
    # KPI 4: Search Brain existence
    kpi4 = "PASS" if 'clean_content' in df.columns and df['clean_content'].iloc[0] != "" else "FAIL"

    print(f"[1] Brand Recovery:      {kpi1}")
    print(f"[2] Price Normalization: {kpi2}")
    print(f"[3] RAM Fill Rate (>80%): {kpi3}")
    print(f"[4] Search Content:      {kpi4}")
    
    return all(x == "PASS" for x in [kpi1, kpi2, kpi3, kpi4])

In [ ]:
# --- CELL 31: Execute Audit ---
if check_kpis(df_clean):
    print("\n ALL SYSTEMS GO. Proceeding to Export.")
    
    # --- CELL 32: Export ---
    output_path = '../Data/processed'
    if not os.path.exists(output_path):
        os.makedirs(output_path)
        print(f" Created folder: {output_path}")
        
    df_clean.to_csv(f'{output_path}/clean_products.csv', index=False)
    print(f" Saved {len(df_clean)} rows to {output_path}/clean_products.csv")
else:
    print("\n AUDIT FAILED. Review the FAILures above before exporting.")

---  FINAL DATA QUALITY AUDIT 🏁 ---
[1] Brand Recovery:      PASS
[2] Price Normalization: PASS
[3] RAM Fill Rate (>80%): PASS
[4] Search Content:      PASS

 ALL SYSTEMS GO. Proceeding to Export.
📁 Created folder: ../Data/processed
💾 Saved 1666 rows to ../Data/processed/clean_products.csv


moving to recomemndation system and egin  with our processed data